In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
%pip uninstall -y axolotl peft transformers accelerate datasets trl optimum cut-cross-entropy flash-attn
%pip install --no-build-isolation git+https://github.com/OpenAccess-AI-Collective/axolotl.git
%pip install --no-build-isolation axolotl[flash-attn]>=0.9.1
%pip install "cut-cross-entropy[transformers] @ git+https://github.com/axolotl-ai-cloud/ml-cross-entropy.git@318b7e2"
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
%pip uninstall -y axolotl peft transformers accelerate datasets trl optimum cut-cross-entropy flash-attn
%pip install --no-build-isolation git+https://github.com/OpenAccess-AI-Collective/axolotl.git
%pip install --no-build-isolation axolotl[flash-attn]>=0.9.1
%pip install "cut-cross-entropy[transformers] @ git+https://github.com/axolotl-ai-cloud/ml-cross-entropy.git@318b7e2"

In [ ]:
!pip install numpy==2.4.2 scipy==1.16.3 scikit-learn==1.6.1 --upgrade --force-reinstall

In [1]:

dataset_id = "winglian/pirate-ultrachat-10k"
uploaded = {}

In [2]:
from axolotl.utils import set_pytorch_cuda_alloc_conf
set_pytorch_cuda_alloc_conf()

In [3]:
from axolotl.cli.config import load_cfg
from axolotl.utils.dict import DictDefault



[2026-03-06 04:59:15,645] [WARNING] [torchao] Skipping import of cpp extensions due to incompatible torch version 2.9.0+cu126 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [7]:
config = DictDefault(
    base_model="Qwen/Qwen2.5-3B-Instruct",
    load_in_4bit=True,
    adapter="qlora",
    lora_r=32,
    lora_alpha=64,
    lora_target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "down_proj",
        "up_proj",
    ],
    lora_qkv_kernel=False,
    lora_o_kernel=False,
    lora_mlp_kernel=False,
    embeddings_skip_upcast=True,
    xformers_attention=True,
    plugins=[],
    sample_packing=False,
    learning_rate=0.00019,
    sequence_len=1024,
    micro_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },
    optimizer="paged_adamw_8bit",
    lr_scheduler="cosine",
    warmup_steps=5,
    fp16=True,
    bf16=False,
    max_grad_norm=0.1,
    num_epochs=1,
    saves_per_epoch=2,
    logging_steps=1,
    output_dir="./outputs/qwen-sft-pirate-rrr",
    chat_template="qwen3",
    datasets=[
        {
            "path": dataset_id,
            "type": "chat_template",
            "split": "train",
            "eot_tokens": ["<|im_end|>"],
        }
    ],
    dataloader_prefetch_factor=None,
    dataloader_num_workers=0,
    dataloader_pin_memory=True,
)
cfg = load_cfg(config)

In [8]:
from axolotl.common.datasets import load_datasets

dataset_meta=load_datasets(cfg=cfg)

Fetching 0 files: 0it [00:00, ?it/s]

Dropping Invalid Sequences (<None or >1024) (num_proc=4):   0%|          | 0/9985 [00:00<?, ? examples/s]

Saving the dataset (0/4 shards):   0%|          | 0/8840 [00:00<?, ? examples/s]

In [11]:
cfg.max_steps=5

In [12]:
from axolotl.train import train
# cfg["trainer"].update({
#     "dataloader_num_workers": 0,
#     "dataloader_prefetch_factor": None,
# })
model,tokenizer,trainer=train(cfg=cfg,dataset_meta=dataset_meta)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Step,Training Loss
1,0.000000
2,0.000000
3,0.000000
4,0.000000
5,0.000000


In [13]:
from transformers import TextStreamer

messages = [
    {
        "role": "user",
        "content": "Explain the Pythagorean theorem to me.",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    enable_thinking=False,
)

outputs = model.generate(
    **tokenizer(prompt, return_tensors="pt").to("cuda"),
    max_new_tokens=192,
    temperature=1.0,
    top_p=0.8,
    top_k=32,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

In [15]:
print(tokenizer.decode(outputs[0],skip_special_tokens=True))

In [17]:

!ls -lh "./outputs/qwen-sft-pirate-rrr"

In [18]:
from huggingface_hub import notebook_login

# remove the partial epoch checkpoints
!rm -rf "./outputs/qwen-sft-pirate-rrr/checkpoint-*"

# HF Notebook login widget
notebook_login()

# upload the LoRA adapter for your model to HF, remember to update the username/model-name below
!huggingface-cli upload --repo-type=model winglian/pirate-qwen-14B "./outputs/qwen-sft-pirate-rrr"